In [47]:
import zipfile
import os
import pandas as pd

In [48]:
#!git clone https://github.com/manikantavs01/DeepLearning_Hackaton.git


fatal: destination path 'DeepLearning_Hackaton' already exists and is not an empty directory.


In [49]:
# Step 3: Extract the zip file
# Create a directory for extraction
extract_dir = "/content/DeepLearning_Hackaton/train"
os.makedirs(extract_dir, exist_ok=True)

# Open and extract the zip file
with zipfile.ZipFile("/content/DeepLearning_Hackaton/train.zip", 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Contents extracted to {extract_dir}")

Contents extracted to /content/DeepLearning_Hackaton/train


In [84]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.utils.data import random_split
from sklearn.model_selection import train_test_split
from PIL import Image
from torchvision.models import resnet18
from torchvision.models import resnet50

# Set device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [51]:
device

device(type='cpu')

In [120]:
train=pd.read_csv("/content/DeepLearning_Hackaton/train/train.csv")
test=pd.read_csv("/content/DeepLearning_Hackaton/sample_submission.csv")

In [121]:
train_paths=train['image_names'].values
train_emer=train['emergency_or_not'].values
test_paths=test['image_names'].values
test_emer=test['emergency_or_not'].values

In [126]:
# Define transformations for the training and test datasets
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    #transforms.RandomResizedCrop(128),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet mean/std
])

In [127]:
# Function to load and preprocess images
def load_images(file_paths, labels, transform):
    images = []
    directory = '/content/DeepLearning_Hackaton/train/images'
    for file_path in file_paths:
        file_path = os.path.join(directory, str(file_path))
        image = Image.open(file_path).convert('RGB')
        image = transform(image)
        images.append(image)
    return torch.stack(images), torch.tensor(labels)

In [128]:
# Load datasets
train_images, train_labels = load_images(train_paths, train_emer, transform)
test_images, test_labels = load_images(test_paths, test_emer, transform)
train_dataset = TensorDataset(train_images, train_labels)
test_dataset = TensorDataset(test_images, test_labels)

In [103]:
# Split training data into training and validation sets
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_data, val_data = random_split(train_dataset, [train_size, val_size])

In [104]:
# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [105]:
model = resnet50(pretrained=True)  # Using pre-trained ResNet50
num_ftrs = model.fc.in_features  # The number of input features to the final fully connected layer
model.fc = nn.Linear(num_ftrs, 1)  # Binary classification

In [106]:
# Define the loss function and optimizer
criterion = nn.BCEWithLogitsLoss()  # Binary cross-entropy loss with logits
optimizer = optim.Adam(model.parameters(), lr=0.001)

def calculate_accuracy(outputs, labels):
    preds = torch.round(torch.sigmoid(outputs))
    correct = (preds == labels).float()
    accuracy = correct.sum() / len(correct)
    return accuracy

In [107]:
# Training the model
num_epochs = 10
for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    running_accuracy = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)

        labels = labels.float().unsqueeze(1)  # Ensure labels are the correct shape
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Track the loss and accuracy
        running_loss += loss.item()
        running_accuracy += calculate_accuracy(outputs, labels).item()

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = 100*running_accuracy / len(train_loader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%')

Epoch [1/10], Loss: 0.4905, Accuracy: 77.90%
Epoch [2/10], Loss: 0.3345, Accuracy: 86.83%
Epoch [3/10], Loss: 0.2283, Accuracy: 91.22%
Epoch [4/10], Loss: 0.1490, Accuracy: 94.79%
Epoch [5/10], Loss: 0.1471, Accuracy: 94.12%
Epoch [6/10], Loss: 0.1369, Accuracy: 94.57%
Epoch [7/10], Loss: 0.1896, Accuracy: 92.11%
Epoch [8/10], Loss: 0.1083, Accuracy: 95.91%
Epoch [9/10], Loss: 0.0763, Accuracy: 97.02%
Epoch [10/10], Loss: 0.0971, Accuracy: 96.35%


In [108]:
# Evaluate the model on the validation set
model.eval()  # Set the model to evaluation mode

running_val_accuracy = 0
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        labels = labels.float().unsqueeze(1)  # Ensure labels are the correct shape
        running_val_accuracy += calculate_accuracy(outputs, labels).item()

val_accuracy = 100*running_val_accuracy / len(val_loader)
print(f'Validation Accuracy: {val_accuracy:.2f}%')

Validation Accuracy: 92.33%


In [129]:
# Evaluate on the test set
model.eval()
running_test_accuracy = 0.0
output_test= []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        labels = labels.float().unsqueeze(1)# Ensure labels are the correct shape
        running_test_accuracy +=calculate_accuracy(outputs, labels).item()
        output_test.append(torch.round(torch.sigmoid(outputs))) #converting sigmoid probabilities to
test_accuracy=100*running_test_accuracy/len(test_loader)
print(f'test Accuracy: {test_accuracy:.2f}%')

test Accuracy: 62.09%


In [130]:
output_test[0].size()
flattened_tensors = [tensor.view(-1, 1) for tensor in output_test]
single_column_tensor = torch.cat(flattened_tensors, dim=0)

# Convert the tensor to a DataFrame
df = pd.DataFrame(single_column_tensor.numpy(), columns=['emergency_or_not'])
# Combine the selected columns into a new DataFrame
combined_df = pd.DataFrame({'image_names': test['image_names'], 'emergency_or_not': df['emergency_or_not']})

# Write the DataFrame to an Excel file
combined_df.to_excel('test.xlsx', index=False)

print("List of tensors has been written to tensor_list_output.xlsx")

List of tensors has been written to tensor_list_output.xlsx
